<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>


<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
}
</style>

# IMU Estimation and Drift

The IMU reports acceleration and angular velocity with measurement errors even while the Duckiedrone is stationary. This notebook uses those measurements to explain attitude estimation and why dead-reckoned position drifts.

By the end of the notebook you should be able to answer:

- How can the stationary proper-acceleration measurement correct roll and pitch, and why does it not provide an absolute yaw reference?

- What does integrating the proper-acceleration measurement twice give, and why is that estimate not usable by itself for position?

- Why can a longer calibration reduce drift without eliminating it?

<span id="why-the-imu-data-matters"></span>

## Why the IMU data matters

The proper-acceleration and angular-velocity measurements together are enough to work out which way the Duckiedrone is tilted, and that tilt is all the flight controller needs to stay level, because a level Duckiedrone has no roll or pitch. The same measurements could in principle also provide the position of the Duckiedrone through double integration, but in practice that estimate drifts away within seconds.

Neither sensor reports attitude directly. The flight controller derives it using an algorithm (an estimator called the __Extended Kalman Filter 2 (EKF2)__) that combines the calibrated proper-acceleration and angular-velocity measurements into an educated guess ("estimate") of the Duckiedrone's orientation. That estimate is good for roll and pitch, but not for yaw.

Two words need separating before that sentence can mean anything.

An __absolute__ measurement is one taken against a reference that exists outside the Duckiedrone and does not move with it, so it can be trusted no matter how long the Duckiedrone has been flying. For example, coordinates from the Global Positioning System (GPS) would provide an absolute position measurement. A __relative__ measurement only describes a change from wherever the Duckiedrone happened to be, so its usefulness depends entirely on knowing that starting point and on nothing having gone wrong since. For example, saying the Duckiedrone has traveled $10\,\mathrm{m}$ in a certain direction, with respect to its takeoff location, is a relative measurement.

Supporting a stationary Duckiedrone against gravitational acceleration produces an upward proper-acceleration measurement, which supplies a directional reference for roll and pitch. It has magnitude $9.81\,\mathrm{m/s^2}$ and points opposite the local gravitational acceleration, which points toward the center of the Earth. Tilt the Duckiedrone and that measurement redistributes across the three sensor axes in a way that depends on the tilt. Thus, the estimator can correct roll and pitch from this stationary proper-acceleration measurement rather than relying on accumulated gyroscope error.

Yaw has no equivalent directional cue. Rotating the Duckiedrone about the local vertical leaves the stationary proper-acceleration measurement identical at every heading, so it cannot tell one heading from another. A magnetometer would supply an outside reference by reading the Earth's magnetic field, but the Duckiedrone does not carry one.

Thus, yaw comes only from integrating the gyroscope, which is the office chair again. Any error in the measured rate is added into the total and keeps accumulating, so the heading estimate drifts slowly away from the truth.

## What an IMU does not measure

Given the above, an IMU alone cannot directly measure:

- __Position__: where the Duckiedrone is;

- __Linear velocity__: how fast it is moving;

- __Absolute yaw__: unless we add a magnetometer to the IMU.

Position, being the double integral of linear acceleration, is recoverable in theory. As [Equation (5) in Notebook 1](./1-imu-basics.ipynb#equation-5) shows, after correcting the proper-acceleration measurement for sensor error, rotating it into the world frame, and adding gravitational acceleration, integrating the resulting estimate of $\mathbf{a}_{\mathrm{W}}$ twice gives position. Estimating position this way, by integrating a measurement of motion with no outside reference to correct it, is called __dead reckoning__.

Dead reckoning in general is not very useful, because the estimate tends to "quickly" diverge from the real value due to noise in the measurements. Every physical sensor is noisy, with no way around it: the electronics add a little noise of their own, the reading is rounded off when it is turned into a number, and the whole thing shifts as the sensor warms up. As [Equation (3) in Notebook 1](./1-imu-basics.ipynb#equation-3) shows, the measured proper acceleration contains proper acceleration, accelerometer bias, and random proper-acceleration measurement noise.

### Measuring noise and bias

[Table 1](#table-1) summarizes $1500$ consecutive messages, $30\,\mathrm{s}$ at about $50\,\mathrm{Hz}$, recorded from the same motionless, level Duckiedrone as the [sample message in Notebook 1](./1-imu-basics.ipynb#gathering-imu-measurements-from-a-duckiedrone). For these stationary comparisons, $\tilde{\mathbf{f}}_{\mathrm{S}}$ and $\tilde{\boldsymbol{\omega}}_{\mathrm{S}}$ denote the measured sensor-frame vectors with the time argument suppressed. A level, motionless sensor has proper acceleration:

$$
\mathbf{f}_{\mathrm{S}}
=
\begin{bmatrix}
0 \\
0 \\
+9.81
\end{bmatrix}
\,\mathrm{m/s^2}.
$$

The measured proper acceleration is approximately:

$$
\tilde{\mathbf{f}}_{\mathrm{S}}
\approx \mathbf{f}_{\mathrm{S}}.
$$

The measured angular velocity is approximately:

$$
\tilde{\boldsymbol{\omega}}_{\mathrm{S}}
\approx
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
\,\mathrm{rad/s}.
$$

[Table 1](#table-1) summarizes the measured statistics from 1500 stationary IMU samples.

<a id="table-1"></a>

Table 1: Summary statistics for $1500$ stationary IMU measurements.

| Quantity | Units | Mean | Standard deviation | Minimum | Maximum |
| --- | --- | --- | --- | --- | --- |
| $\tilde{f}_x(t_k)$ | $\mathrm{m/s^2}$ | $-0.009$ | $0.024$ | $-0.156$ | $+0.248$ |
| $\tilde{f}_y(t_k)$ | $\mathrm{m/s^2}$ | $-0.093$ | $0.022$ | $-0.297$ | $+0.019$ |
| $\tilde{f}_z(t_k)$ | $\mathrm{m/s^2}$ | $+9.801$ | $0.116$ | $+9.479$ | $+10.166$ |
| $\tilde{\omega}_x(t_k)$ | $\mathrm{rad/s}$ | $+0.00002$ | $0.00041$ | $-0.0016$ | $+0.0017$ |
| $\tilde{\omega}_y(t_k)$ | $\mathrm{rad/s}$ | $+0.00001$ | $0.00057$ | $-0.0016$ | $+0.0017$ |
| $\tilde{\omega}_z(t_k)$ | $\mathrm{rad/s}$ | $+0.00005$ | $0.00052$ | $-0.0016$ | $+0.0017$ |

Two different effects are visible in [Table 1](#table-1):

- Individual vertical readings are spread from $9.479\,\mathrm{m/s^2}$ to $10.166\,\mathrm{m/s^2}$ while the
  Duckiedrone does not move at all. The [sample message in Notebook 1](./1-imu-basics.ipynb#gathering-imu-measurements-from-a-duckiedrone) happened to read $9.917\,\mathrm{m/s^2}$.
  The random measurement-noise component $n_{a,z}$ is close to zero on average, so __averaging removes it__: the
  mean of $9.801\,\mathrm{m/s^2}$ is within $0.006\,\mathrm{m/s^2}$ of standard gravity, $9.80665\,\mathrm{m/s^2}$.

- The mean has a persistent offset that averaging does not remove. The mean of the
  $\tilde{f}_y(t_k)$ samples is $-0.093\,\mathrm{m/s^2}$, and it stays there however many samples
  are added. Some of that offset comes from the bench not being perfectly level; the sensor's
  roughly constant contribution is the accelerometer __bias__, written $b_{a,i}$ for axis $i$. It is a
  consequence of the sensor being a physical object, not an extra property that IMUs happen to
  have.

### Why the bias is fatal to dead reckoning

Now integrate a constant bias twice. Let $b = b_{a,i}$ be the accelerometer bias on one axis. It contributes a velocity error growing as $b\,t$. Writing $\varepsilon(t)$ for the resulting position error, with zero initial position and velocity error, gives:

<span id="equation-1"></span>

$$
\varepsilon(t) = \tfrac{1}{2}\,b\,t^{2}.
\tag{1}
$$

The velocity error is linear in time and the position error quadratic in time. Nothing in the dead-reckoning calculation can tell that offset apart from genuine motion, because the two produce identical readings.

Let $g_0 = 9.80665\,\mathrm{m/s^2}$ denote standard gravity. Datasheets quote accelerometer bias in milligravity units ($\mathrm{m}g_0$), where $1\,\mathrm{m}g_0 = 10^{-3}g_0 \approx 0.01\,\mathrm{m/s^2}$. The $-0.093\,\mathrm{m/s^2}$ measured mean for the $\tilde{f}_y(t_k)$ samples above is about $9\,\mathrm{m}g_0$. A bias of a few milligravity units is unremarkable for a sensor of this class, and it sounds harmless.

It is not harmless, and the quickest way to see why is to watch it happen.

### Activity 1: an ideal Duckiedrone with one flaw

<a id="activity-1"></a>

This first demonstration is deliberately idealized, so that exactly one thing can go wrong. A Duckiedrone sits on the floor of a $2.5\,\mathrm{m}$ room, powered up and completely still. It never moves. Assume it already knows its orientation and has perfectly accounted for gravitational acceleration. Thus, the only defect left is a small constant bias $b = b_{a,z}$ on the vertical axis. It dead-reckons its altitude by integrating that reading twice, as [Equation (1)](#equation-1) describes.

Green is where the Duckiedrone really is. Red is where it believes it is.

Start at $5\,\mathrm{m}g_0$ and drag the time slider. Then try $1\,\mathrm{m}g_0$, which would be an excellent sensor, and see whether it helps.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import FloatSlider, interact

# Variables use the notation introduced above:
#   b     Vertical accelerometer bias [m/s^2].
#   t_k   Discrete sample times [s].
#   x     Dead-reckoned altitude [m].

G = 9.80665  # Standard-gravity magnitude [m/s^2].
SAMPLE_RATE = 50.0  # Illustrative IMU sample rate [Hz].
TRUE_ALT = 0.0  # Ground-truth altitude [m].
CEILING = 2.5  # Room height [m].


def ideal_drift(bias_mg=5.0, duration_s=60.0):
    """Compare where the Duckiedrone really is with where dead reckoning says it is."""
    b = bias_mg * G / 1000.0  # Convert milli-g to m/s^2.

    # An IMU reports at discrete instants, so build the sample times explicitly.
    t_k = np.arange(0.0, duration_s, 1.0 / SAMPLE_RATE)

    # A constant bias gives the closed-form double integral x = (1/2) b t^2.
    x = TRUE_ALT + 0.5 * b * t_k**2

    fig, ax = plt.subplots(figsize=(9, 5))

    ax.axhspan(
        CEILING, 10, color="0.92"
    )  # Shade the region above the ceiling.
    ax.axhline(CEILING, color="0.5", lw=1)
    ax.text(
        duration_s * 0.99,
        CEILING + 0.06,
        "ceiling",
        ha="right",
        fontsize=9,
        color="0.4",
    )

    # Ground-truth altitude.
    ax.axhline(TRUE_ALT, color="tab:green", lw=2.5, label="where it really is")

    # Plot a subset of discrete estimates so their markers remain distinguishable.
    stride = max(1, len(t_k) // 120)
    marker_gap = stride / SAMPLE_RATE
    ax.plot(
        t_k[::stride],
        x[::stride],
        color="tab:red",
        linestyle="none",
        marker=".",
        markersize=6,
        label=f"dead-reckoned altitude estimate (one dot per {marker_gap:.1f} s)",
    )

    ax.set_ylim(-0.15, CEILING + 0.5)
    ax.set_yticks(np.arange(0.0, CEILING + 0.5, 0.5))
    ax.set_xlim(0, duration_s)
    ax.set_xlabel("time [s]")
    ax.set_ylabel("altitude [m]")
    ax.legend(loc="center right")

    ax.set_title(
        f"bias b = {bias_mg:.1f} mg = {b:.3f} m/s^2    "
        f"after {t_k[-1]:.0f} s it is off by {x[-1] - TRUE_ALT:.1f} m"
    )
    fig.tight_layout()
    plt.show()


interact(
    ideal_drift,
    bias_mg=FloatSlider(
        value=5.0,
        min=0.1,
        max=5.0,
        step=0.1,
        description="bias [mg]",
        continuous_update=False,
    ),
    duration_s=FloatSlider(
        value=60.0,
        min=5.0,
        max=300.0,
        step=1.0,
        description="time [s]",
        continuous_update=False,
    ),
);

<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
}
</style>

Reducing the bias delays that moment without changing the outcome, because the error grows with the square of time however small $b$ is.

A fair objection is that the plot only demonstrates [Equation (1)](#equation-1), because that relation is what produced it. The next demonstration replaces the idealized calculation with a recording.

### Activity 2: the same experiment on a physical Duckiedrone

<a id="activity-2"></a>

A physical Duckiedrone was placed on the floor of a room, powered up with its motors off, and left static for $5\,\mathrm{min}$. Its proper-acceleration measurements were logged throughout at about $50\,\mathrm{Hz}$, and that log ships with this learning experience as [assets/data/imu_still_5min.csv](../assets/data/imu_still_5min.csv). These are recorded sensor measurements, not simulated ones. The red dots are the altitude obtained by double integrating every sample, though only every few samples is plotted so the dots stay distinguishable. The green line is where the Duckiedrone really is, which is known continuously: it never moved.

Measurements arrive at discrete instants rather than continuously. Let $k$ index the global samples. Thus, the double integral in [Equation (5) in Notebook 1](./1-imu-basics.ipynb#equation-5) becomes nested sums over samples. Because this experiment keeps the Duckiedrone level, let $\tilde{f}_{\mathrm{S},z,k}$ be the measured component along its sensor $z$ axis at the global sample time $t_k$. It is a noisy proper-acceleration measurement of the proper-acceleration component $f_{\mathrm{S},z,k}$. Let $\Delta t_k$ be the time since the previous sample, $\Delta t_k = t_k - t_{k-1}$. Let $N \in \mathbb{Z}_{>0}$ be the number of calibration samples in a window of duration $T_{\mathrm{cal}}$ (in $\mathrm{s}$). The procedure is:

1. Hold the Duckiedrone still and collect the samples
   $\tilde{f}_{\mathrm{S},z,0}, \ldots, \tilde{f}_{\mathrm{S},z,N-1}$ during $T_{\mathrm{cal}}$.

2. Average them into a stationary reference $\hat{r}$:

   $$
   \hat{r} = \frac{1}{N}\sum_{k=0}^{N-1} \tilde{f}_{\mathrm{S},z,k}.
   $$

   This single number combines the proper acceleration produced by supporting the stationary
   Duckiedrone against gravitational acceleration with the bias. A static recording cannot
   separate the two, because both are constant offsets while the sensor is not moving.

3. For every later sample, $k \geq N$, form the baseline-corrected vertical acceleration
   estimate $\hat{a}_{z,k}$:

   $$
   \hat{a}_{z,k} = \tilde{f}_{\mathrm{S},z,k} - \hat{r}.
   $$

   Because this experiment keeps the Duckiedrone level, $\hat{a}_{z,k}$ estimates its
   world-frame vertical acceleration $a_{\mathrm{W},z,k}$. For a tilted or moving Duckiedrone,
   one sensor-axis reading is not enough: the full proper-acceleration measurement
   $\tilde{\mathbf{f}}_{\mathrm{S}}$ must first be corrected for sensor error, rotated into the
   world frame, and have the gravitational-acceleration term $\mathbf{g}_{\mathrm{W}}$ added
   back. For the stationary Duckiedrone here, $a_{\mathrm{W},z,k}$ is zero. Thus, any nonzero
   $\hat{a}_{z,k}$ is estimation error.

4. Let $v_k$ and $x_k$ be the vertical velocity and altitude at sample $k$. Integration begins at
   sample $k = N$, with $v_N = x_N = 0$. For $k \geq N + 1$, update the vertical velocity with
   the trapezoidal rule:

<span id="equation-2"></span>

   $$
   v_k = v_{k-1}
   + \tfrac{1}{2}\left(\hat{a}_{z,k-1} + \hat{a}_{z,k}\right)\Delta t_k.
   \tag{2}
   $$

   Then update the altitude:

<span id="equation-3"></span>

   $$
   x_k = x_{k-1} + \tfrac{1}{2}\left(v_{k-1} + v_k\right)\Delta t_k.
   \tag{3}
   $$

   Velocity is in $\mathrm{m/s}$ and altitude in $\mathrm{m}$. The code below implements [Equation (2)](#equation-2) and [Equation (3)](#equation-3) directly.

The first slider sets $T_{\mathrm{cal}}$. The second sets how long the estimate then runs. For the plot, the post-calibration samples are indexed locally by $j \in \mathbb{Z}_{\geq 0}$. Let $\tau_j$ be the time since calibration, so $\tau_j = t_{N+j} - t_N$. The code stores $\tau_j$ in an array named `t_k`. Thus $\tau_0 = 0$ corresponds to the global sample time $t_N$, and $\tau_j - \tau_{j-1} = \Delta t_{N+j}$. Consequently, changing the time origin does not change the integration.

A longer calibration reduces the random error in $\hat{r}$, but it cannot separate sensor bias from the proper acceleration caused by support against gravitational acceleration or remove a bias that changes between calibration and integration.

Try to find a calibration time that keeps the red dots inside the room, given a default ceiling of $2.5\,\mathrm{m}$.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt  # noqa: F811
import numpy as np  # noqa: F811
from ipywidgets import FloatSlider, interact

# Variables use the notation from the procedure above:
#   f_z_tilde_all  Recorded vertical proper-acceleration samples [m/s^2].
#   t_all          Recording times [s].
#   r_hat          Stationary reference [m/s^2].
#   a_z_hat        Baseline-corrected vertical acceleration [m/s^2].
#   t_k            Elapsed time since integration began [s].
#   v, x           Dead-reckoned velocity [m/s] and altitude [m].

CEILING = 2.5  # Room height [m].
FLOOR = 0.0  # Ground-truth altitude [m].

# Load five minutes of vertical proper-acceleration data from a stationary Duckiedrone.
# Columns: t [s], ax, ay, az [m/s^2].
data_path = Path("assets/data/imu_still_5min.csv")
if not data_path.exists():
    data_path = Path("../assets/data/imu_still_5min.csv")
data = np.genfromtxt(data_path, delimiter=",", names=True)
t_all, f_z_tilde_all = data["t"], data["az"]


def real_drift(calibration_s=10.0, watch_s=60.0):
    """Dead reckon altitude from the recording, after calibrating for so many seconds."""
    # The mean includes sensor bias and the proper acceleration produced by
    # supporting the stationary Duckiedrone against gravitational acceleration.
    r_hat = f_z_tilde_all[t_all < calibration_s].mean()

    # Select post-calibration samples and measure time from the first one.
    integrating = (t_all >= calibration_s) & (t_all <= calibration_s + watch_s)
    sample_times = t_all[integrating]
    t_k = sample_times - sample_times[0]

    # Remove the stationary reference; the result is ideally zero while level and still.
    a_z_hat = f_z_tilde_all[integrating] - r_hat

    # Approximate the two integrals with the trapezoidal rule.
    dt = np.diff(t_k)
    v = np.concatenate(
        ([0.0], np.cumsum(0.5 * (a_z_hat[:-1] + a_z_hat[1:]) * dt))
    )
    x = FLOOR + np.concatenate(([0.0], np.cumsum(0.5 * (v[:-1] + v[1:]) * dt)))

    fig, ax = plt.subplots(figsize=(9, 5))

    ax.axhspan(CEILING, 10, color="0.92")
    ax.axhspan(-10, FLOOR, color="0.88")
    ax.axhline(CEILING, color="0.5", lw=1)
    ax.text(
        t_k[-1] * 0.99,
        CEILING + 0.06,
        "ceiling",
        ha="right",
        fontsize=9,
        color="0.4",
    )

    # Ground-truth altitude.
    ax.axhline(FLOOR, color="tab:green", lw=2.5, label="where it really is")

    # Plot a subset of discrete estimates so their markers remain distinguishable.
    stride = max(1, len(t_k) // 120)
    marker_gap = stride * float(np.median(dt))
    ax.plot(
        t_k[::stride],
        x[::stride],
        color="tab:red",
        linestyle="none",
        marker=".",
        markersize=6,
        label=f"dead-reckoned altitude estimate (one dot per {marker_gap:.1f} s)",
    )

    ax.set_ylim(-1.0, CEILING + 0.5)
    ax.set_xlim(0, t_k[-1])
    ax.set_xlabel("time since integration started [s]")
    ax.set_ylabel("altitude [m]")
    ax.legend(loc="upper right")

    n_calib = len(t_all[t_all < calibration_s])
    ax.set_title(
        f"calibrated for {calibration_s:.0f} s over {n_calib} samples    "
        f"after {t_k[-1]:.0f} s of dead reckoning it is off by {x[-1] - FLOOR:+.1f} m"
    )
    fig.tight_layout()
    plt.show()


interact(
    real_drift,
    calibration_s=FloatSlider(
        value=10.0,
        min=1.0,
        max=120.0,
        step=1.0,
        description="calibrate [s]",
        continuous_update=False,
    ),
    watch_s=FloatSlider(
        value=60.0,
        min=10.0,
        max=180.0,
        step=5.0,
        description="watch [s]",
        continuous_update=False,
    ),
);

<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
}
</style>

### What these two toy examples show

$10\,\mathrm{s}$ of calibration is already longer than anyone waits in practice, and the Duckiedrone still puts itself through the ceiling within $20\,\mathrm{s}$ of starting to integrate. Increasing the calibration time changes the drift without removing it: average a full $2\,\mathrm{min}$ of a Duckiedrone that was sitting still, and over the $3\,\mathrm{min}$ of recording that remain the estimate ends up about $2.4\,\mathrm{m}$ below the floor.

That second result is the one worth pausing on, because it shows the problem is not a bias that was measured carelessly. Averaged over the full $5\,\mathrm{min}$, the stationary vertical reading is within $0.1\,\mathrm{m}g_0$ of standard gravity, which is smaller than the smallest bias the first slider will accept. The overall average is excellent. What it hides is variation between short windows: the value measured during a $10\,\mathrm{s}$ calibration is already different from the value in the $60\,\mathrm{s}$ to $70\,\mathrm{s}$ window, and double integration turns that small mismatch into meters of position error.

Thus, a longer calibration can delay the failure but never removes it. The culprit is not the sensor but the integration, which has no way to tell a small constant offset from real motion and no outside reference to check itself against.

## Further reading

[Duckiedrone sensor calibration](https://docs.duckietown.com/ente/opmanual-dd24/30-duckiedrone-setup-and-config/50-sensor-calibrations.html) covers the physical calibration behind the measurement errors discussed here. [Notebook 3](./3-rotation-representations.ipynb) develops the rotations needed to express IMU readings in other frames.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()